In [1]:
import json

In [2]:
llms = ["gemma2_9b", "llama3.1_8b", "mistral_7b", "qwen2.5_7b"]
rms = ["fsfairx_rm", "mistral_rm"]

llm_mapping = {
    "gemma2_9b": "google/gemma-2-9b-it", 
    "llama3.1_8b": "meta-llama/Llama-3.1-8B-Instruct",
    "llama3.2_3b": "meta-llama/Llama-3.2-3B-Instruct",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3",
    "qwen2.5_7b": "Qwen/Qwen2.5-7B-Instruct",
    "qwen3_4b": "Qwen/Qwen3-4B-Instruct-2507",
}

rm_mapping = {
    "fsfairx_rm": "sfairXC/FsfairX-LLaMA3-RM-v0.1", 
    "mistral_rm": "weqweasdas/RM-Mistral-7B"
}

In [3]:
dataset = "alpaca"
rm = "fsfairx_rm"
llm = "mistral_7b"
distribution = "shifted_exponential"
transformation = "cdf"
batch_size = "1"
alpha = "0.99"

data = []
with open(f"../slurm_result_{dataset}/result_1_{rm}_{llm}_{distribution}_{transformation}_bs{batch_size}_a{alpha}.json") as f:
    data = json.load(f)

In [4]:
data.keys()

dict_keys(['llm_name', 'rm_name', 'distribution', 'transformation', 'batch_size', 'epoch', 'alpha', 'delta', 'costs', 'per_prompt'])

In [8]:
data["per_prompt"][0].keys()

dict_keys(['prompt_index', 'pandora', 'fixed_n'])

In [7]:
data["per_prompt"][0]["pandora"]

[{'cost': 0.02, 'median_utility': 0.06852652606687447},
 {'cost': 0.01, 'median_utility': 0.18907056617779708},
 {'cost': 0.008, 'median_utility': 0.21474730543724793},
 {'cost': 0.006, 'median_utility': 0.22907056617779709},
 {'cost': 0.004, 'median_utility': 0.27453933300114963},
 {'cost': 0.002, 'median_utility': 0.30689526542660406},
 {'cost': 0.001, 'median_utility': 0.3274851868626199},
 {'cost': 0.0008, 'median_utility': 0.3397318784483148},
 {'cost': 0.0006, 'median_utility': 0.3600002077698289},
 {'cost': 0.0004, 'median_utility': 0.3692392613764981},
 {'cost': 0.0002, 'median_utility': 0.4072689279872028},
 {'cost': 0.0001, 'median_utility': 0.4370851927252567}]

In [11]:
data["per_prompt"][0]["fixed_n"][0].keys()

dict_keys(['cost', 'results'])

In [12]:
data["per_prompt"][0]["fixed_n"][0]["results"][:10]

[{'n': 1, 'mean_utility': 0.03263211252828265},
 {'n': 2, 'mean_utility': 0.03688383593817874},
 {'n': 3, 'mean_utility': 0.056357807370171174},
 {'n': 4, 'mean_utility': 0.05170762933029478},
 {'n': 5, 'mean_utility': 0.0560234066674031},
 {'n': 6, 'mean_utility': 0.07255325966840732},
 {'n': 7, 'mean_utility': 0.06642484691345749},
 {'n': 8, 'mean_utility': 0.06170977624314476},
 {'n': 9, 'mean_utility': 0.0615780849846878},
 {'n': 10, 'mean_utility': 0.05433921336978985}]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_revenue_comparison(costs_shortlist, NAs, As, K, factor, llm_name=None, rm_name=None):
    """
    Create improved visualization comparing non-adaptive and Pandora's Box algorithms.
    
    Parameters:
    -----------
    costs_shortlist : list
        List of tuples (index, cost) for different cost scenarios
    NAs : list
        Non-adaptive algorithm results
    As : list
        Adaptive (Pandora's Box) algorithm results
    K : int
        Number of iterations/samples
    factor : float
        Factor for reward calculation
    """

    llm_label = llm_mapping.get(llm_name, llm_name)
    rm_label = rm_mapping.get(rm_name, rm_name)
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # Main Title
    fig.text(0.5, 0.98, 'Profit Comparison: Non-Adaptive vs Pandora\'s Box Algorithm', 
             fontsize=16, fontweight='bold', ha='center', va='top')
    
    # --- CHANGE 1: Combined the two subtitles into a single line ---
    fig.text(0.5, 0.92, f'LLM: {llm_label}  |  Reward Model: {rm_label} | Dataset: AlpacaEval', 
             fontsize=16, ha='center', va='top')
    
    # Store statistics for summary
    stats = []
    
    # First pass to calculate common y-axis limits for consistency
    y_min, y_max = float('inf'), float('-inf')
    
    for i, c in costs_shortlist:
        ys = []
        for j in range(len(NAs[0])):
            y = np.mean([NAs[l][j]["mean"] * factor - NAs[l][j]["sample_count"] * c for l in range(K)])
            ys.append(y)
        ystar = np.mean([As[l][i]["revenue"] for l in range(K)])
        
        current_min = min(min(ys), ystar)
        current_max = max(max(ys), ystar)
        y_min = min(y_min, current_min)
        y_max = max(y_max, current_max)
    
    # Add padding to y-axis
    y_padding = (y_max - y_min) * 0.1
    y_min -= y_padding
    y_max += y_padding

    res = []
    
    # Create each subplot
    for idx, (i, c) in enumerate(costs_shortlist):
        ax = axes[idx]
        
        # Calculate data points for non-adaptive algorithm
        xs = []
        ys = []
        for j in range(len(NAs[0])):
            x = np.median([NAs[l][j]["sample_count"] for l in range(K)])
            y = np.median([(NAs[l][j]["mean"] - NAs[l][j]["sample_count"]*c) for l in range(K)])
            xs.append(x)
            ys.append(y)
        
        # Calculate Pandora's Box performance
        ystar = np.median([As[l][i]["revenue"] for l in range(K)])
        
        # Calculate max values
        max_na = np.max(ys)
        adaptive_scaled = ystar
        
        # Store statistics
        optimal_samples = xs[np.argmax(ys)]
        stats.append({
            'cost': c,
            'pandora': ystar,
            'max_non_adaptive': max_na,
            'optimal_samples': optimal_samples,
            'reward_cost_ratio': 0.5 / c
        })
        
        # Plot non-adaptive points (blue dots forming a curve)
        ax.scatter(xs, ys, s=20, c='blue', alpha=0.7, 
                   label='Non-Adaptive' if idx == 0 else "")
        
        # Connect dots with a line for better visualization
        sorted_indices = np.argsort(xs)
        ax.plot([xs[i] for i in sorted_indices], 
                [ys[i] for i in sorted_indices], 
                'b-', alpha=0.3, linewidth=1)
        
        # Plot Pandora's Box performance (red horizontal line)
        ax.axhline(y=adaptive_scaled, color='red', linestyle='--', linewidth=2, 
                   label='Pandora\'s Box' if idx == 0 else "")
        
        # Mark the optimal point for non-adaptive
        ax.scatter([optimal_samples], [max_na], s=100, c='green', 
                   marker='*', zorder=5, 
                   label='Optimal Non-Adaptive' if idx == 0 else "")
        
        # Add text annotations for max values (bottom right corner)
        ax.text(0.98, 0.15, f'Non-Adaptive Max: {max_na:.3f}', 
                transform=ax.transAxes, fontsize=9, ha='right',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))
        
        ax.text(0.98, 0.05, f'Adaptive Max: {adaptive_scaled:.3f}', 
                transform=ax.transAxes, fontsize=9, ha='right',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.7))

        res.append(adaptive_scaled/max_na)
        
        # Formatting
        ax.set_ylim(-1, +1)
        ax.set_xlabel('Sample Count', fontsize=14)
        if idx == 0:
            ax.set_ylabel('Profit (Utility - Total Cost)', fontsize=16)
        
        # Add title with cost and reward/cost ratio
        ax.set_title(f'Cost= {c}', 
                     fontsize=16, pad=10)
        
        # Grid for better readability
        ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
        ax.set_facecolor('#f9f9f9')
        
        # Add shading to show positive/negative revenue regions
        ax.axhspan(0, +1, alpha=0.1, color='green')
        ax.axhspan(-1, 0, alpha=0.1, color='red')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
        
        # Format tick labels
        ax.tick_params(axis='both', which='major', labelsize=14)
        
    # --- CHANGE 2: Moved the legend to be vertical and on the right-hand side ---
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.98, 0.5), 
               ncol=1, frameon=True, fontsize=16, title='Algorithm Type')
    
    # --- CHANGE 3: Adjusted layout to make space for the right-hand legend ---
    # The 'rect' argument [left, bottom, right, top] is adjusted. 
    # Reducing the 'right' value from 1.0 to 0.97 creates space for the legend.
    # The 'top' value is increased slightly as there is more vertical space now.
    plt.tight_layout(rect=[0, 0, 0.97, 0.85]) 
    
    plt.show()
    return fig, stats, res